# Experiment: Agentic Real-Time Metadata Standardization

## Configurations

Run this notebook from the repository root, which is where it lives — every path below is relative to it.

Before running it, create a `.env` file in the project root with the following API keys:

```
OPENAI_API_KEY=your-openai-api-key          # Required — for LLM calls
CEDAR_API_KEY=your-cedar-api-key            # Required — for fetching CEDAR templates
BIOPORTAL_API_KEY=your-bioportal-api-key    # Required — for ontology term lookups

LANGFUSE_PUBLIC_KEY=pk-lf-your-public-key   # Optional — enables observability for LLM calls and agent runs
LANGFUSE_SECRET_KEY=sk-lf-your-secret-key   # Optional — tracing is on only when both keys are set
LANGFUSE_HOST=https://cloud.langfuse.com    # Optional — use https://us.cloud.langfuse.com for US cloud
```

See [.env.example](.env.example) for a template.

In [ ]:
import sys
from pathlib import Path

EVALUATION_DIR = Path.cwd() / "evaluation"
if str(EVALUATION_DIR) not in sys.path:
    sys.path.insert(0, str(EVALUATION_DIR))

`DATA_ROOT` points to the root data directory. The evaluation functions expect the following directory structure underneath it:

```
DATA_ROOT/
├── schemas/
│   ├── atacseq.json              # JSON Schema for each assay type
│   ├── lcms.json
│   └── ...
├── atacseq/                      # One directory per assay type
│   ├── input/
│   │   ├── atacseq-<hash>.json   # Legacy metadata records (input)
│   │   └── ...
│   ├── gold/
│   │   ├── atacseq-<hash>.json   # Gold-standard reference outputs
│   │   └── ...
│   └── output/
│       └── <MODEL>/              # e.g., "gpt-5.6-terra"
│           ├── baseline/
│           │   ├── atacseq-<hash>.json   # Prompt-only LLM outputs
│           │   └── ...
│           └── arms-agent/
│               ├── atacseq-<hash>.json   # Tool-augmented agent outputs
│               └── ...
└── ...
```

Gold-standard and output files share the same filenames so that each output can be matched to its reference for evaluation.

The leaf directory is the run's name, and it is the same name the CLI takes as the value of its workflow flag: `python -m evaluation --prompt-only baseline` or `--agent-tool arms-agent` writes to the matching directory under `--output`, so a run made from the command line and one made from this notebook land in the same place.

In [ ]:
DATA_ROOT = "data"

`MODEL` specifies which LLM model was used for the migration run.

In [ ]:
MODEL = "gpt-5.6-terra"

`ASSAYS` and `RUN_TYPES` select what the sweep covers: every assay in `ASSAYS` is run through every condition in `RUN_TYPES`, so the sweep is the product of the two. Trim either list to run a subset.

`RUN_TYPES` takes the same values as the CLI's workflow flags, and each names the directory its predictions are written to:

| `RUN_TYPES` entry | CLI equivalent |
|---|---|
| `baseline` | `--prompt-only baseline` — field and vocabulary names only |
| `arms-agent` | `--agent-tool arms-agent` — the tool-augmented agent |

In [ ]:
# Every assay the sweep covers, in the order it runs them.  Trim the list to run a
# subset -- ["atacseq"] to validate one assay before spending on the rest.
ASSAYS = [
    "atacseq",
    "rnaseq",
    "af",
    "celldive",
    "codex",
    "histology",
    "imc-2d",
    "lightsheet",
    "mibi",
    "desi",
    "lcms",
    "maldi",
]

# Every condition each of those assays is run through.
RUN_TYPES = ["baseline", "arms-agent"]

## Run Experiments

Two calls: `plan_sweep` says what the sweep covers and checks that it can run, `run_sweep` runs it. Nothing is spent until `RUN_EXPERIMENT` is `True`.

Every assay in `ASSAYS` goes through every condition in `RUN_TYPES`: One assay at a time, all of its conditions before the next assay starts. Each run reads `DATA_ROOT/<assay>/input` and writes to `DATA_ROOT/<assay>/output/<MODEL>/<RUN_TYPE>`.

In [ ]:
from sweep import plan_sweep, run_sweep

RUN_EXPERIMENT = False  # Change to "True" to run the experiment. Be careful it costs money to run it!

### The plan

`plan_sweep` reads the API keys from `.env`, then raises on anything that would otherwise fail partway through a sweep: an unknown assay, an unknown condition, an assay with no input records, or a key the chosen conditions need and the environment does not have. This pre-check ensures that the experiment settings are valid before incurring any costs.

In [ ]:
plan = plan_sweep(DATA_ROOT, MODEL, assays=ASSAYS, run_types=RUN_TYPES)

### The sweep

The only cell that spends money, and only when `RUN_EXPERIMENT` is `True`. While it is `False` the call will just make a list of runs that it would make. Each run is handed to a worker thread. The sweep adds no parallelism of its own: it waits for each run to finish before starting the next.

In [ ]:
run_sweep(plan, dry_run=not RUN_EXPERIMENT)

## Data Analysis

Data analysis reads only files already on disk--no API keys, no LLM calls--so this section can be re-run freely.

In [ ]:
import pandas as pd

import notebook_utils
from analysis.significance import (
    build_per_assay_precision_recall_table,
    build_precision_recall_table,
)

# The two conditions compared: the baseline to beat, and the system challenging it.
BASELINE_RUN = "baseline"
SYSTEM_RUN = "arms-agent"

### Precision and recall

Two metrics reported for every assay and every field category.

- **Precision** — of the values a method filled in, how many were right. Low precision means it makes things up.
- **Recall** — of the values the gold standard asks for, how many the method produced. Low recall means it leaves things out.

Blank fields earn no credit either way. Agreeing that a field should be empty is not counted, so a method cannot score well by filling in nothing.

**The field categories.** (1) `Ontology-constrained fields` are the ones the CEDAR template ties to a controlled vocabulary, so the method has to come back with a term from the right ontology. (2) `Non-ontology-constrained fields` are everything else: free text, numbers, identifiers.

### Units of analysis

The corpus has repeated field-value pairs across many records. One systematic fix can therefore be counted a hundred times, which lead to same predictions are scored multiple times.

| Scoring type | Data unit | What it answers |
| --- | --- | --- |
| **Field-level** | One field of one record | How much work a curator is spared |
| **Deduplicated** | One distinct field-value | How wide a range of values a method handles |

The two can disagree, and the gap is informative. A gain that shows up in field-level but not deduplicated is a real saved work but it may not mean the method is robust in handling a wider range of values.

### Field-level scoring

Field-level scoring treats each field in each record as a separate unit, reflecting the correction workload: how many individual values are correctly rectified across the entire corpus.

**Reading the scoring table**

1. One row per assay and field type, with an `All assays` block pooled over the whole corpus at the end.

2. On result columns per method, each cell reads `point [low, high]`: the score, then the 95% confidence interval around it. Intervals come from resampling whole records. The `difference` column is the arms method minus the baseline method. `n_records` is how many gold/prediction pairs stand behind the row.

In [ ]:
COLUMNS = ["assay", "category", "metric", "n_records", BASELINE_RUN, SYSTEM_RUN, "difference"]

# One row per assay and field type.
per_assay = build_per_assay_precision_recall_table(DATA_ROOT, MODEL, baseline_run=BASELINE_RUN, system_run=SYSTEM_RUN)

# The same scores pooled over every assay.
pooled = build_precision_recall_table(DATA_ROOT, MODEL, baseline_run=BASELINE_RUN, system_run=SYSTEM_RUN)
pooled = pooled[pooled["metric"].isin(("precision", "recall"))].assign(assay="All assays")

instance_weighted = pd.concat([per_assay[COLUMNS], pooled[COLUMNS]], ignore_index=True)
print(instance_weighted.to_string(index=False))

### Deduplicated scoring

Deduplicated scoring removes repeated occurrences of the same values across records, reflecting performance across the range of values encountered rather than their frequency. For recall, repeated reference-standard field–value pairs are counted once, and recall is calculated across these distinct pairs. For precision, repeated asserted field–value pairs are counted once, and precision is then averaged across fields.

Hypothesis Testing: **does ARMS handle a wider range of values than the baseline, or could the gap be an accident of which records happened to land in the corpus?**

- **H₀** — the two methods are interchangeable. Any difference is sampling noise.
- **H₁** — ARMS differs from the baseline on that metric.
- **Direction** — every number is *ARMS minus baseline*, so positive favours ARMS.

**Reading the scoring table**

1. One row per field type and metric, pooled over the whole corpus. Both methods' deduplicated scores sit beside the difference and its p-value, so every tested quantity appears next to its own test.

2. `difference [95% CI]` says how big the gap is; `p_value` says how surprising it would be under H₀. The two should agree: An interval clear of 0 goes with p < 0.05. When they disagree the case is borderline, and the honest reading is "not established".

3. Read `paired on` and `n_items` beside them: they name what one unit was, and how many of them stand behind the row.

In [ ]:
deduplicated = notebook_utils.show_deduplicated_tests(
    DATA_ROOT,
    MODEL,
    baseline_run=BASELINE_RUN,
    system_run=SYSTEM_RUN,
)

## Plots

### Precisions and Recall

Precision and recall by assay type and field category, weighted by field instance. Each panel is one assay type, with the number of records in parentheses; recall is on the horizontal axis and precision on the vertical axis. Open markers are the prompt-only LLM baseline and filled markers the ARMS agent, and marker shape denotes the field category. Points nearer the upper right are better. Confidence intervals are omitted for legibility and are reported in the text where they are discussed.

In [ ]:
from plots import plot_pr_space

plot_pr_space(
    DATA_ROOT,
    MODEL,
    assays=tuple(ASSAYS),
    baseline_runs=("baseline",),
    system_runs=("arms-agent",),
    field_types=("ontology", "non_ontology", "all"),
    shared_window=True,
    error_axes=False,
    show_f1_contours=False,
    no_color=True,
)

## Error Analysis

A **category** is the disagreement case, named for what the run did: `substitutions`, `omissions`, `insertions`. Its **sub-categories** split them into specific cases.

| Category | Sub-category | Description |
|---|---|---|
| `substitutions` | `close_match` | Gold has a value, ARMS asserts one that is partly right without being it exactly. |
| `substitutions` | `mislocate_legacy_value` | Gold has a value, ARMS asserts a value the legacy record carries but under a *wrong* field. The instinct to copy was right but the source field was wrong. |
| `substitutions` | `completely_wrong` | Gold has a value, ARMS asserts one that is neither near gold's nor carried anywhere in the legacy record. Nothing in the input accounts for it. |
| `omissions` | `underestimate_legacy_value` | Gold has a value the legacy record carries, ARMS asserts nothing. It was there to be copied and was left behind. |
| `omissions` | `entirely_dont_know` | Gold has a value the legacy record does not carry, ARMS asserts nothing. There was nothing in the input to go on, and ARMS did not make any guess. |
| `insertions` | `overestimate_legacy_value` | Gold leaves the field blank, ARMS asserts a value the legacy record carries. It read more into the legacy record than the curator did. |
| `insertions` | `too_optimistic_answer` | Gold leaves the field blank, ARMS asserts a value of its own, the legacy record carries nothing. The agent answered where it should have abstained. |

In [ ]:
import notebook_utils

SYSTEM_RUN = "arms-agent"

errors = notebook_utils.show_error_analysis(DATA_ROOT, MODEL, SYSTEM_RUN, apply_dedup=True)